# 05. 딥러닝 (PyTorch) — 시험 13~14번

> **배우는 것**: 텐서 · `TensorDataset` / `DataLoader` · `nn.Sequential` · `nn.Linear` · `nn.ReLU` · `nn.Dropout` · 출력층과 손실함수 짝 맞추기 · `zero_grad → backward → step` 학습 루프 · `train()` / `eval()` / `no_grad()` · history 기록과 학습 곡선 · 조기 종료 직접 구현 · 예측 후처리(`sigmoid > 0.5`, `argmax`) · **시험 13~14번 템플릿**
> ⏱ 예상 소요시간: 80분  |  📑 같이 볼 슬라이드: `slides/05_딥러닝.html`

> 📝 **시험 13~14번 형식**: "지정된 구조로 모델 만들기 → 학습하며 `history` 에 loss / val_loss 기록 → 학습 곡선 그리기 → 검증 지표 계산".
> 이 노트북 마지막 6번 섹션의 **시험형 템플릿**이 그 흐름 그대로예요. 변수명 규약: `model`(nn.Module), `history = {'loss': [...], 'val_loss': [...]}`, 예측은 `model.eval()` + `torch.no_grad()`.

### 🔑 가장 중요한 표 — 출력층과 손실함수의 짝
| 문제 | 마지막 층 | 손실함수 | 예측 후처리 |
|---|---|---|---|
| **회귀** (숫자) | `nn.Linear(…, 1)` | `nn.MSELoss()` | 그대로 |
| **이진 분류** (0/1) | `nn.Linear(…, 1)` (sigmoid 없음!) | `nn.BCEWithLogitsLoss()` ← sigmoid가 손실 **안에** 들어 있음 | `torch.sigmoid(out) > 0.5` |
| **다중 분류** (0,1,2 정수) | `nn.Linear(…, 클래스 수)` (softmax 없음!) | `nn.CrossEntropyLoss()` ← softmax가 손실 **안에** 들어 있음 | `out.argmax(dim=1)` |

### 학습 루프 5줄 = 시험 공부 루틴
```python
for xb, yb in train_loader:     # 문제집을 32문제씩 끊어서
    optimizer.zero_grad()       # ① 지난번 채점 흔적 지우기
    out = model(xb)             # ② 문제 풀기 (순전파)
    loss = loss_fn(out, yb)     # ③ 채점하기 (손실)
    loss.backward()             # ④ 어디서 틀렸는지 거꾸로 추적 (역전파)
    optimizer.step()            # ⑤ 공부법 고치기 (가중치 업데이트)
```

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'   # macOS에서 XGBoost·PyTorch 동시 사용 시 멈춤 방지
# ▶ 맨 처음 한 번 실행하세요 (데이터 경로 + 자동 채점 도구 준비)
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data/'          # 데이터 폴더 위치
import aice_check as check     # check.p01_q1() 처럼 쓰는 채점 도구

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# --- 03 에서 배운 전처리를 한 번에 (그냥 실행) ---
df = pd.read_csv(DATA_DIR + 'telco_churn.csv')
df = df.drop(['customerID', 'DeviceProtection'], axis=1)            # 불필요 / 결측 많은 컬럼
df = df[df['MonthlyCharges'] < 200].dropna()                        # 이상치, 결측 행
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = LabelEncoder().fit_transform(df['Churn'])             # No=0, Yes=1
df_preset = pd.get_dummies(df, columns=df.select_dtypes('object').columns, dtype=int)

X = df_preset.drop('Churn', axis=1)
y = df_preset['Churn']
feature_names = X.columns                                           # 스케일링하면 이름이 사라지니 저장
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

n_features = X_train.shape[1]
print('입력 특성 수:', n_features, '| X_train', X_train.shape, type(X_train).__name__, X_train.dtype)

## 1. 텐서와 DataLoader — 데이터를 PyTorch가 먹는 모양으로

- **텐서(tensor)** = GPU에도 올라가는 numpy 배열이에요. `torch.tensor(넘파이배열, dtype=...)`로 바꿔요.
- **dtype 규칙**: 입력 X는 `torch.float32`. 정답 y는 **회귀·이진분류면 `float32` + 모양 `(N, 1)`**, **다중분류면 `torch.long` + 모양 `(N,)`**.
- **DataLoader** = 데이터를 `batch_size`개씩 잘라 주는 급식 배식기예요. 학습 때는 `shuffle=True`로 매 epoch 순서를 섞어요.

### ✏️ 문제 1. import
`torch`, `torch.nn`(별칭 **`nn`**), 그리고 `torch.utils.data` 에서 **`TensorDataset`**, **`DataLoader`** 를 불러오세요.

In [ ]:
import ___
import torch.nn as ___
from torch.utils.data import ___, ___
print(torch.__version__)

<details><summary>💡 힌트 (막히면 클릭)</summary>

`import torch.nn as nn` 은 PyTorch 코드의 국룰 별칭이에요. pandas 의 `pd` 같은 것.

</details>

In [ ]:
check.p05_q1()

### ✏️ 문제 2. numpy → 텐서
- `X_train`, `X_test` → **`Xtr`**, **`Xte`** (dtype **float32**)
- `y_train`, `y_test` (pandas Series) → **`ytr`**, **`yte`** (dtype **float32**, 모양 **(N, 1)**)

> 이진분류 손실 `BCEWithLogitsLoss` 는 모델 출력 `(N, 1)` 과 정답의 모양이 **같아야** 해요. 그래서 `reshape(-1, 1)` 로 세로로 세워요.

In [ ]:
Xtr = torch.tensor(X_train, dtype=___)
Xte = torch.tensor(X_test, dtype=torch.float32)
ytr = torch.tensor(y_train.to_numpy(), dtype=torch.float32).___
yte = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(___)
Xtr.shape, ytr.shape, Xtr.dtype

<details><summary>💡 힌트 (막히면 클릭)</summary>

`dtype=torch.float32`, `.reshape(-1, 1)` — -1 은 '나머지는 알아서' 라는 뜻이에요.

</details>

In [ ]:
check.p05_q2()

### ✏️ 문제 3. TensorDataset + DataLoader
- `Xtr, ytr` 를 묶어 **`train_ds`** 를 만드세요.
- **`train_loader`**: batch_size **32**, **shuffle=True**

In [ ]:
train_ds = ___
train_loader = DataLoader(train_ds, batch_size=___, shuffle=___)

xb, yb = next(iter(train_loader))     # 첫 배치 하나만 꺼내 보기
xb.shape, yb.shape

<details><summary>💡 힌트 (막히면 클릭)</summary>

`TensorDataset(X, y)` 는 같은 번호의 X 와 y 를 짝지어 줘요. 배치 모양은 `(32, 특성 수)`, `(32, 1)`.

</details>

In [ ]:
check.p05_q3()

## 2. 이진 분류 모델 (고객 이탈 0/1)

`nn.Sequential` 은 층을 **순서대로** 통과시키는 상자예요.
- `nn.Linear(입력 수, 출력 수)` — 완전연결층. **입력 수를 직접** 적어야 해요 (앞 층 출력 수 = 다음 층 입력 수).
- 활성화 함수 `nn.ReLU()` 는 **별도 층**으로 넣어요.
- `nn.Dropout(0.3)` — 학습 때마다 뉴런 30% 를 랜덤으로 쉬게 해서 **과적합 방지**.
- 출력층은 **`nn.Linear(32, 1)` 에서 끝** — sigmoid 는 손실함수 `BCEWithLogitsLoss` 안에 들어 있어요.

### ✏️ 문제 4. 모델 쌓기
`torch.manual_seed(42)` 를 먼저 실행하고, 아래 구조로 **`model`** 을 만드세요.
`Linear(n_features → 64)` → ReLU → Dropout(**0.3**) → `Linear(64 → 32)` → ReLU → `Linear(32 → 1)`

In [ ]:
torch.manual_seed(42)                 # 결과 재현용
model = nn.Sequential(
    nn.Linear(___, 64),
    ___,
    nn.Dropout(___),
    nn.Linear(64, ___),
    nn.ReLU(),
    nn.Linear(32, ___),
)
model

<details><summary>💡 힌트 (막히면 클릭)</summary>

앞 층의 출력 수 = 다음 층의 입력 수 (64 → 64, 32 → 32). 마지막에 `nn.Sigmoid()` 를 넣지 마세요!

</details>

In [ ]:
check.p05_q4()

### ✏️ 문제 5. 손실함수와 옵티마이저
- **`loss_fn`**: 이진분류용 **`nn.BCEWithLogitsLoss()`**
- **`optimizer`**: **Adam**, 학습률 **lr=0.001**, 학습할 대상은 `model.parameters()`

In [ ]:
loss_fn = ___
optimizer = torch.optim.___(___, lr=___)

<details><summary>💡 힌트 (막히면 클릭)</summary>

손실함수는 '채점 기준', 옵티마이저는 '공부법'. lr(학습률)은 한 번에 얼마나 크게 고칠지예요.

</details>

In [ ]:
check.p05_q5()

### ✏️ 문제 6. 한 epoch 학습 함수
**`train_one_epoch(model, loader, loss_fn, optimizer)`** 의 빈칸을 채우세요. 데이터 전체를 한 바퀴 돌며 학습하고 **평균 손실**을 돌려줘요.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.___()                         # 학습 모드 (Dropout 켜짐)
    total = 0.0
    for xb, yb in loader:
        optimizer.___()              # ① 지난 기울기 지우기
        out = model(xb)                      # ② 순전파
        loss = loss_fn(___)            # ③ 손실 (예측, 정답 순서)
        loss.___()                    # ④ 역전파
        optimizer.___()                   # ⑤ 가중치 업데이트
        total += loss.___() * len(xb)     # 텐서 → 파이썬 숫자
    return total / len(loader.dataset)

<details><summary>💡 힌트 (막히면 클릭)</summary>

순서 외우기: **zero_grad → forward → loss → backward → step**. `.item()` 은 숫자 하나짜리 텐서를 float 로 꺼내요.

</details>

In [ ]:
check.p05_q6()

### ✏️ 문제 7. 평가 함수
**`evaluate(model, X, y, loss_fn)`**: 평가 모드로 바꾸고, 기울기 계산 없이 손실을 계산해 **float** 로 돌려주세요.

In [ ]:
def evaluate(model, X, y, loss_fn):
    model.___()                          # 평가 모드 (Dropout 꺼짐)
    with torch.___():                  # 기울기 계산 끄기 (빠르고 메모리 절약)
        out = model(X)
        return loss_fn(out, y).item()

<details><summary>💡 힌트 (막히면 클릭)</summary>

`model.eval()` + `with torch.no_grad():` 는 항상 세트예요. 예측·평가할 때마다 이 두 줄로 시작하세요.

</details>

In [ ]:
check.p05_q7()

### ✏️ 문제 8. 학습 + history 기록 + 조기 종료
최대 **30 epoch** 학습하면서
- 매 epoch 의 학습 손실 / 검증 손실을 **`history`** 딕셔너리(`'loss'`, `'val_loss'` 리스트)에 기록하고
- 검증 손실이 **5 epoch 연속** 좋아지지 않으면 멈추고 (**`patience = 5`**)
- 가장 좋았던 가중치(**`best_state`**)로 되돌리세요.

> '5번 연속 모의고사 점수가 안 오르면 공부를 멈추고, 가장 잘 봤던 날의 상태로 돌아간다' 는 규칙이에요.

In [ ]:
import copy

history = {'loss': [], 'val_loss': []}
best_val, best_state, patience, wait = float('inf'), None, ___, 0

for epoch in range(___):
    tr_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    va_loss = evaluate(model, Xte, yte, loss_fn)
    history['loss'].append(tr_loss)
    history[___].append(va_loss)

    if va_loss < ___:                              # 좋아졌다 → 기록 갱신
        best_val, wait = va_loss, 0
        best_state = copy.deepcopy(model.___)  # 가중치 사진 찍어 두기
    else:
        wait += 1
        if wait >= patience:
            print(f'{epoch + 1} epoch 에서 조기 종료')
            break

model.___(best_state)                      # 최고 시점으로 되돌리기
print(f'best val_loss = {best_val:.4f}')

<details><summary>💡 힌트 (막히면 클릭)</summary>

`copy.deepcopy` 를 안 쓰면 '사진'이 아니라 '실시간 영상'을 저장하는 셈이라 계속 바뀌어요. 되돌릴 때는 `model.load_state_dict(best_state)`.

</details>

In [ ]:
check.p05_q8()

### ✏️ 문제 9. 학습 곡선 그리기
`history['loss']`, `history['val_loss']` 를 그리고 title **'Loss'**, xlabel **'Epochs'**, legend **['loss', 'val_loss']** 를 붙이세요. 그린 축은 **`ax9`** 에 저장하세요.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history[___])
plt.plot(history['val_loss'])
plt.title('Loss'); plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(___)
ax9 = plt.gca()
plt.show()

<details><summary>💡 힌트 (막히면 클릭)</summary>

val_loss 가 바닥을 찍고 다시 올라가기 시작하면 = 과적합 신호. 조기 종료가 그 바닥에서 멈춰 줘요.

</details>

In [ ]:
check.p05_q9()

### ✏️ 문제 10. 예측 → 0/1 → 정확도
- 평가 모드 + `no_grad` 로 `Xte` 를 넣어 **로짓**을 구하고
- `torch.sigmoid` 로 확률 → **`proba`** (1차원 numpy)
- 0.5 기준으로 0/1 → **`pred`** (int)
- **`acc`** = `accuracy_score(y_test, pred)`

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
with torch.no_grad():
    logits = model(Xte)
proba = torch.___(logits).numpy().___()
pred = (proba > ___).astype(int)
acc = accuracy_score(___)
print('정확도:', round(acc, 3))

<details><summary>💡 힌트 (막히면 클릭)</summary>

모델 출력은 확률이 아니라 **로짓**(아무 실수)이에요. `torch.sigmoid` 를 거쳐야 0~1 확률이 돼요.

</details>

In [ ]:
check.p05_q10()

## 3. 회귀 모델 (아파트 가격, 단위: 억)

이진분류와 **딱 두 군데**만 달라요: 손실함수 `nn.MSELoss()`, 그리고 예측값을 **그대로** 써요. 출력층은 똑같이 `nn.Linear(…, 1)`.

In [ ]:
# --- 아파트 가격 데이터 전처리 (그냥 실행) ---
house = pd.read_csv(DATA_DIR + 'house_price.csv')
house = house[house['area_m2'] < 300]                               # 면적 입력 오류 제거
house['built_year'] = house['built_year'].fillna(house['built_year'].median())
house['station_dist_m'] = house['station_dist_m'].fillna(house['station_dist_m'].median())
house_preset = pd.get_dummies(house, columns=['region', 'parking'], dtype=int)
from sklearn.preprocessing import StandardScaler

Xh = house_preset.drop('price', axis=1)
yh = house_preset['price'] / 10000                 # 만원 → 억 (정답 크기가 너무 크면 학습이 느려요)
Xh_train, Xh_test, yh_train, yh_test = train_test_split(Xh, yh, test_size=0.2, random_state=42)
sc = StandardScaler()
Xh_train = sc.fit_transform(Xh_train)
Xh_test = sc.transform(Xh_test)

Xh_tr = torch.tensor(Xh_train, dtype=torch.float32)
Xh_te = torch.tensor(Xh_test, dtype=torch.float32)
yh_tr = torch.tensor(yh_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
yh_te = torch.tensor(yh_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)
reg_loader = DataLoader(TensorDataset(Xh_tr, yh_tr), batch_size=32, shuffle=True)
print('회귀 입력:', Xh_tr.shape, '| 정답:', yh_tr.shape)

### ✏️ 문제 11. 회귀 모델 + 손실
`torch.manual_seed(42)` 후
- **`reg_model`**: `Linear(입력 수 → 64)` → ReLU → `Linear(64 → 1)`
- **`reg_loss_fn`**: **`nn.MSELoss()`**
- **`reg_opt`**: Adam, **lr=0.01**

In [ ]:
torch.manual_seed(42)
reg_model = nn.Sequential(
    nn.Linear(Xh_tr.shape[___], 64),
    nn.ReLU(),
    nn.Linear(64, ___),
)
reg_loss_fn = ___
reg_opt = torch.optim.Adam(reg_model.parameters(), lr=___)

<details><summary>💡 힌트 (막히면 클릭)</summary>

회귀는 출력층에 활성화 함수를 붙이지 않아요. 입력 수는 `Xh_tr.shape[1]`.

</details>

In [ ]:
check.p05_q11()

### ✏️ 문제 12. 회귀 학습 + MAE
위에서 만든 `train_one_epoch`, `evaluate` 를 **그대로 재사용**해 **50 epoch** 학습하고 손실을 **`reg_history`**(`'loss'`, `'val_loss'`) 에 기록하세요.
그다음 `Xh_te` 예측값으로 MAE 를 계산해 **`reg_mae`** 에 저장하세요.

In [ ]:
from sklearn.metrics import mean_absolute_error

reg_history = {'loss': [], 'val_loss': []}
for epoch in range(___):
    reg_history['loss'].append(train_one_epoch(reg_model, ___, reg_loss_fn, reg_opt))
    reg_history['val_loss'].append(evaluate(reg_model, Xh_te, yh_te, ___))

reg_model.eval()
with torch.no_grad():
    reg_pred = reg_model(Xh_te).numpy().flatten()
reg_mae = mean_absolute_error(___, reg_pred)
print(f'MAE = {reg_mae:.3f}억')

<details><summary>💡 힌트 (막히면 클릭)</summary>

함수를 만들어 두면 모델·로더·손실만 바꿔 끼우면 돼요. 이게 PyTorch 학습 루프의 장점이에요.

</details>

In [ ]:
check.p05_q12()

## 4. 다중 분류 (와인 품종 3종류, sklearn 내장 데이터)

다중분류는 **정답 dtype 이 다르다**는 게 핵심이에요.
- 정답 y: **`torch.long`**(정수), 모양 **`(N,)`** — reshape 하지 않아요!
- 출력층: `nn.Linear(…, 클래스 수)` · 손실: `nn.CrossEntropyLoss()` (softmax 내장)
- 예측: `out.argmax(dim=1)` — 가장 점수가 큰 클래스 번호

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    wine.data, wine.target, test_size=0.2, stratify=wine.target, random_state=42)
scm = StandardScaler()
Xm_train = scm.fit_transform(Xm_train)
Xm_test = scm.transform(Xm_test)
print('특성 수:', Xm_train.shape[1], '| 클래스:', np.unique(wine.target))

### ✏️ 문제 13. 다중분류 텐서 (dtype 주의!)
- **`Xm_tr`**, **`Xm_te`**: float32
- **`ym_tr`**, **`ym_te`**: **`torch.long`**, 모양 `(N,)` 그대로

In [ ]:
Xm_tr = torch.tensor(Xm_train, dtype=torch.float32)
Xm_te = torch.tensor(Xm_test, dtype=torch.float32)
ym_tr = torch.tensor(ym_train, dtype=___)
ym_te = torch.tensor(ym_test, dtype=torch.long)
ym_tr.shape, ym_tr.dtype

<details><summary>💡 힌트 (막히면 클릭)</summary>

`CrossEntropyLoss` 는 정답이 **클래스 번호(정수)** 여야 해요. float 이나 (N,1) 모양을 넣으면 에러가 나요.

</details>

In [ ]:
check.p05_q13()

### ✏️ 문제 14. 다중분류 모델 + 학습
`torch.manual_seed(42)` 후 **`mc_model`**: `Linear(13 → 32)` → ReLU → `Linear(32 → 3)`, 손실 **`mc_loss_fn = nn.CrossEntropyLoss()`**, Adam lr=0.01.
`train_one_epoch` 로 **100 epoch** 학습하세요 (batch_size 16).

In [ ]:
torch.manual_seed(42)
mc_model = nn.Sequential(nn.Linear(13, 32), nn.ReLU(), nn.Linear(32, ___))
mc_loss_fn = nn.___()
mc_opt = torch.optim.Adam(mc_model.parameters(), lr=0.01)
mc_loader = DataLoader(TensorDataset(Xm_tr, ym_tr), batch_size=___, shuffle=True)

for epoch in range(100):
    mc_last = train_one_epoch(mc_model, mc_loader, mc_loss_fn, mc_opt)
print('마지막 학습 손실:', round(mc_last, 4))

<details><summary>💡 힌트 (막히면 클릭)</summary>

출력 수 = 클래스 수 = 3. softmax 층은 넣지 않아요 (손실 안에 있음).

</details>

In [ ]:
check.p05_q14()

### ✏️ 문제 15. argmax 로 클래스 예측
`Xm_te` 의 예측 클래스 번호를 **`mc_pred`**(1차원 numpy) 에, 정확도를 **`mc_acc`** 에 저장하세요.

In [ ]:
mc_model.eval()
with torch.no_grad():
    scores = mc_model(Xm_te)                      # (N, 3) 클래스별 점수
mc_pred = scores.___(dim=___).numpy()
mc_acc = accuracy_score(ym_test, mc_pred)
print('예측:', mc_pred[:10], '| 정확도:', round(mc_acc, 3))

<details><summary>💡 힌트 (막히면 클릭)</summary>

`dim=1` = 각 행(샘플)마다 3개 열 중 최댓값 위치. numpy 로는 `np.argmax(p, axis=1)`.

</details>

In [ ]:
check.p05_q15()

## 5. 자주 나는 에러 — 일부러 내 보고 메시지 읽는 법 익히기
아래 셀은 **일부러** 틀린 코드를 실행해서 실제 에러 메시지를 보여줘요. 메시지의 키워드(`Float`, `Long`, `size`, `shape`)를 눈에 익혀 두세요.

In [ ]:
tests = {
    '① 다중분류 정답을 float 로': lambda: nn.CrossEntropyLoss()(torch.randn(4, 3), torch.tensor([0., 1., 2., 1.])),
    '② 다중분류 정답을 (N,1) 로 세움': lambda: nn.CrossEntropyLoss()(torch.randn(4, 3), torch.tensor([[0], [1], [2], [1]])),
    '③ 이진분류 정답 모양 (N,) vs 출력 (N,1)': lambda: nn.BCEWithLogitsLoss()(torch.randn(4, 1), torch.tensor([0., 1., 1., 0.])),
    '④ numpy float64 를 그대로 모델에': lambda: nn.Linear(3, 1)(torch.tensor(np.random.rand(2, 3))),
    '⑤ 입력 수가 안 맞는 Linear': lambda: nn.Linear(5, 1)(torch.randn(2, 3)),
}
for name, fn in tests.items():
    try:
        fn()
        print(name, '→ 에러 없음')
    except Exception as e:
        print(f'{name}\n   {type(e).__name__}: {str(e).splitlines()[0][:120]}')

| 에러 키워드 | 원인 | 해결 |
|---|---|---|
| `expected scalar type Long but found Float` | 다중분류 정답이 float | 다중분류 y → `torch.long` |
| `0D or 1D target tensor expected` | 다중분류 정답을 `(N,1)` 로 세움 | 다중분류 y 는 `(N,)` 그대로 |
| `Target size ... must be the same as input size` | 출력 `(N,1)` vs 정답 `(N,)` | `y.reshape(-1, 1)` |
| `mat1 and mat2 must have the same dtype` | float64(Double) 텐서 | `torch.tensor(X, dtype=torch.float32)` |
| `mat1 and mat2 shapes cannot be multiplied` | `Linear` 입력 수 ≠ 특성 수 | `nn.Linear(X.shape[1], …)` |

## 6. 시험형 템플릿 — 13~14번을 PyTorch 로 푸는 정석 흐름

시험 13~14번은 보통 이렇게 나와요.
> **13.** 아래 구조로 모델을 만들고 학습하세요. 결과는 `history` 에 저장하세요.
> **14.** `history` 의 loss / val_loss 로 학습 곡선을 그리고, 검증 데이터의 지표를 계산하세요.

공식 샘플(도착시간 예측, 회귀)을 그대로 옮긴 연습이에요. 데이터는 3번의 아파트 가격(`Xh_tr`, `yh_tr`, `Xh_te`, `yh_te`)을 써요.
**변수명 규약**: `model`(nn.Module) · `history = {'loss': [...], 'val_loss': [...], 'val_mae': [...]}` · 예측은 `model.eval()` + `torch.no_grad()`.

### ✏️ 문제 16. [13번] 지정 구조로 모델 만들기
`torch.manual_seed(42)` 후 **`model`** 을 아래 구조로 만드세요.
`Linear(입력 수 → 128)` → ReLU → Dropout(**0.2**) → `Linear(128 → 64)` → ReLU → `Linear(64 → 1)`
손실 **`loss_fn = nn.MSELoss()`**, **`optimizer`** = Adam(lr=**0.001**), 학습용 **`train_loader`** = batch_size **16**, shuffle=True

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(Xh_tr.shape[1], ___),
    nn.ReLU(),
    nn.___(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(___),
)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_loader = DataLoader(TensorDataset(Xh_tr, yh_tr), batch_size=___, shuffle=True)

<details><summary>💡 힌트 (막히면 클릭)</summary>

시험지의 '128 → Dropout 0.2 → 64 → 1' 을 한 층씩 그대로 옮기면 돼요. 숫자 하나라도 다르면 오답이에요.

</details>

In [ ]:
check.p05_q16()

### ✏️ 문제 17. [13번] 30 epoch 학습 + history 기록
**30 epoch** 학습하면서 매 epoch 마다 **`history`** 에 `'loss'`(학습 평균 손실), `'val_loss'`(검증 손실), `'val_mae'`(검증 MAE)를 기록하세요.

In [ ]:
history = {'loss': [], 'val_loss': [], 'val_mae': []}
for epoch in range(___):
    history['loss'].append(___(model, train_loader, loss_fn, optimizer))
    model.eval()
    with torch.no_grad():
        out = model(Xh_te)
        history['val_loss'].append(loss_fn(out, yh_te).item())
        history['val_mae'].append((out - yh_te).___().mean().item())
print(f"마지막 val_loss {history['val_loss'][-1]:.3f} | val_mae {history['val_mae'][-1]:.3f}억")

<details><summary>💡 힌트 (막히면 클릭)</summary>

MAE = |예측 − 정답| 의 평균 → `(out - y).abs().mean()`. `.item()` 으로 숫자로 꺼내서 리스트에 넣어요.

</details>

In [ ]:
check.p05_q17()

### ✏️ 문제 18. [14번] 학습 곡선 + 검증 지표
- `history['loss']`, `history['val_loss']` 로 그래프를 그리고 title **'Model Loss'**, xlabel **'Epochs'**, ylabel **'Loss'**, legend **['loss', 'val_loss']** 를 붙이세요. 축은 **`ax14`** 에 저장하세요.
- `model.eval()` + `torch.no_grad()` 로 `Xh_te` 를 예측해 **`dnn_mae`**(sklearn `mean_absolute_error`) 를 계산하세요.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history['loss'])
plt.plot(history[___])
plt.title(___); plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(['loss', 'val_loss'])
ax14 = plt.gca()
plt.show()

model.___()
with torch.___():
    dnn_pred = model(Xh_te).numpy().flatten()
dnn_mae = mean_absolute_error(yh_test, dnn_pred)
print(f'DNN MAE = {dnn_mae:.3f}억  (04 머신러닝 실습의 랜덤포레스트와 비교해 보세요)')

<details><summary>💡 힌트 (막히면 클릭)</summary>

그래프 제목·축 이름·범례 문구는 시험지에 적힌 그대로 쓰세요. 예측 전에는 꼭 `model.eval()` (Dropout 끄기).

</details>

In [ ]:
check.p05_q18()

## ✅ 정리 — 딥러닝 PyTorch 체크리스트
| 단계 | 코드 | 놓치기 쉬운 점 |
|---|---|---|
| 텐서 변환 | `torch.tensor(X, dtype=torch.float32)` | 회귀·이진 y 는 `.reshape(-1, 1)`, 다중 y 는 `torch.long` |
| 배치 | `DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)` | 검증 데이터는 shuffle 필요 없음 |
| 모델 | `nn.Sequential(nn.Linear(n,64), nn.ReLU(), nn.Dropout(0.3), ..., nn.Linear(…, 출력 수))` | 앞 층 출력 수 = 다음 층 입력 수 |
| 손실 | 회귀 `MSELoss` · 이진 `BCEWithLogitsLoss` · 다중 `CrossEntropyLoss` | 출력층에 sigmoid / softmax 넣지 않기 |
| 학습 | `zero_grad → model(xb) → loss → backward → step` | `model.train()` 먼저 |
| 기록 | `history = {'loss': [], 'val_loss': []}` | 매 epoch `.item()` 으로 append |
| 조기 종료 | patience + `copy.deepcopy(model.state_dict())` | 끝나면 `load_state_dict(best_state)` |
| 예측 | `model.eval()` + `torch.no_grad()` → `sigmoid > 0.5` / `argmax(dim=1)` | 출력은 확률이 아니라 로짓 |

In [ ]:
check.progress()